# 4.1 Carpotle

Mantener el péndulo en equilibro durante 500 pasos (entorno cartpole v-1)

In [1]:
import os, time
import numpy as np
import torch
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
print("gymnasium:", gym.__version__)

torch: 2.12.0+cu130 | CUDA: False
gymnasium: 1.2.3


/home/jorge/Escritorio/MASTER/AprendizajeRefuerzo/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
env = gym.make("CartPole-v1")
model_zoo = DQN(
    "MlpPolicy", env, verbose=0, seed=42,
    learning_rate=2.3e-3,
    batch_size=64,
    buffer_size=100_000,
    learning_starts=1_000,
    gamma=0.99,
    target_update_interval=10,
    train_freq=256,
    gradient_steps=128,
    exploration_fraction=0.16,
    exploration_final_eps=0.04,
    policy_kwargs=dict(net_arch=[256, 256]),
)
t0 = time.time()
model_zoo.learn(total_timesteps=50_000)
print(f"Entrenamiento Polictica DQN: {time.time()-t0:.1f}s")
mean_r, std_r = evaluate_policy(model_zoo, env, n_eval_episodes=10, deterministic=True)
print(f"Reward Promedio: {mean_r:.1f} +- {std_r:.1f}")
env.close()

Entrenamiento Polictica DQN: 55.1s


/home/jorge/Escritorio/MASTER/AprendizajeRefuerzo/.venv/lib/python3.11/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Reward Promedio: 500.0 +- 0.0


In [3]:
import cv2
env = gym.make("CartPole-v1", render_mode="rgb_array")
frames = []
for ep in range(5):
    obs, _ = env.reset(seed=100+ep)
    done = False
    while not done:
        frames.append(env.render())
        action, _ = model_zoo.predict(obs, deterministic=True)
        obs, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
env.close()

/home/jorge/Escritorio/MASTER/AprendizajeRefuerzo/.venv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [4]:
video_filename = "cartpole_buena_poliy.mp4"
height, width, _ = frames[0].shape  

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video = cv2.VideoWriter(video_filename, fourcc, 30.0, (width, height))

for frame in frames:
    video.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
video.release()

print(f"Video guardado como {video_filename}")
print(f"frames: {len(frames)}")

Video guardado como cartpole_buena_poliy.mp4
frames: 2500


In [5]:
from IPython.display import HTML
from base64 import b64encode
import os

save_path = "cartpole_buena_poliy.mp4"
compressed_path = "result_compressed.mp4"

os.system(f"ffmpeg -i {save_path} -vcodec libx264 {compressed_path}")

mp4 = open(compressed_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=800 controls>
      <source src="%s" type="video/mp4">
</video>""" % data_url)


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab